# LLM Classification Finetuning: Training notebook (llm-cls-train)

**Settings (right panel):** Accelerator = **GPU T4 x2**, Internet = **ON**

**Steps:**
1. First `N_TRAIN = 200` vachu **Run All** (quick test, ~10 min). Error illama kadaisi cell varaikum pogudha nu paaru.
2. Apram `N_TRAIN = 20000` aakki **Save Version → Save & Run All (Commit)**. Background-la run aagum, browser close pannalam.

Output: merged fine-tuned model (`qwen-cls/`), `val.parquet`, bitsandbytes wheel. Inference notebook ithai input-a use pannum.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"      # oru GPU mattum (simple + stable)
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!pip uninstall -y -q torchao   # Kaggle-oda pazhaya torchao, puthu peft-oda clash aagudhu
!pip install -q peft
# Inference notebook internet OFF-la run aagum, so bitsandbytes wheel-a ippove download pannrom
!pip download -q bitsandbytes --no-deps -d /kaggle/working/wheels
!ls -lh /kaggle/working/wheels

import torch, transformers, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
N_TRAIN  = 2000       # 200 = quick test, 2000 = medium test, 20000 = full run (Save & Run All)
N_VAL    = 2000       # validation (log loss check + inference benchmark-ku)
MAX_P, MAX_R = 96, 200   # prompt max 96 tokens, ovvoru response max 200 tokens
OUT  = "/kaggle/working"
SEED = 42

In [ ]:
import glob, json, time
import numpy as np, pandas as pd

train_path = glob.glob("/kaggle/input/**/train.csv", recursive=True)[0]
df = pd.read_csv(train_path)
print(df.shape)
print(df.columns.tolist())

# prompt / response columns JSON list-a irukum (multi-turn), athai plain text aakrom
def parse(x):
    try:
        v = json.loads(x)
        if isinstance(v, list):
            return "\n".join("" if s is None else str(s) for s in v)
        return str(v)
    except Exception:
        return str(x)

for c in ["prompt", "response_a", "response_b"]:
    df[c] = df[c].apply(parse)

# label: 0 = A win, 1 = B win, 2 = tie
df["label"] = df[["winner_model_a", "winner_model_b", "winner_tie"]].values.argmax(1)

df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = df.iloc[:N_VAL].copy()
tr_df  = df.iloc[N_VAL:N_VAL + N_TRAIN].copy()
val_df.to_parquet(f"{OUT}/val.parquet")   # inference notebook benchmark-ku

print("train:", len(tr_df), "| val:", len(val_df))
print(tr_df["label"].value_counts(normalize=True).round(3))

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "right"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

H_P = tok("Prompt: ", add_special_tokens=False)["input_ids"]
H_A = tok("\n\nResponse A: ", add_special_tokens=False)["input_ids"]
H_B = tok("\n\nResponse B: ", add_special_tokens=False)["input_ids"]
H_Q = tok("\n\nWhich response is better?", add_special_tokens=False)["input_ids"]

# Ovvoru part-ayum thani-a truncate pannrom, so response B eppavume cut aagaathu
def build(d):
    P = tok(d["prompt"].tolist(),     add_special_tokens=False, truncation=True, max_length=MAX_P)["input_ids"]
    A = tok(d["response_a"].tolist(), add_special_tokens=False, truncation=True, max_length=MAX_R)["input_ids"]
    B = tok(d["response_b"].tolist(), add_special_tokens=False, truncation=True, max_length=MAX_R)["input_ids"]
    return [H_P + p + H_A + a + H_B + b + H_Q for p, a, b in zip(P, A, B)]

t = time.time()
tr_ids, val_ids = build(tr_df), build(val_df)
print(f"tokenized in {time.time()-t:.1f}s | max len {max(map(len, tr_ids))} | mean len {np.mean(list(map(len, tr_ids))):.0f}")

In [ ]:
from datasets import Dataset
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

tr_ds  = Dataset.from_dict({"input_ids": tr_ids,  "labels": tr_df["label"].tolist()})
val_ds = Dataset.from_dict({"input_ids": val_ids, "labels": val_df["label"].tolist()})

# fp32 weights + fp16 autocast training: T4-la numerically safe
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=3, dtype=torch.float32)
model.config.pad_token_id = tok.pad_token_id
torch.nn.init.zeros_(model.score.weight)   # start = 1/3,1/3,1/3 -> loss ln3 = 1.0986 (overconfident start illa)

lora = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
import math, inspect
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

BS, GA = 4, 4                                        # effective batch = 16
LR_LORA, LR_HEAD = 1e-4, 5e-5                        # head: 2e-4 = blow up, 1e-5 = too slow, 5e-5 = middle
total_steps  = math.ceil(len(tr_ds) / (BS * GA))
warmup_steps = max(10, int(0.05 * total_steps))
log_steps    = max(1, min(25, total_steps // 5))

cfg = dict(
    output_dir=f"{OUT}/ckpt",
    per_device_train_batch_size=BS,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=GA,
    learning_rate=LR_LORA,
    num_train_epochs=1,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    group_by_length=True,
    logging_steps=log_steps,
    save_strategy="steps",
    save_steps=400,
    save_total_limit=1,
    report_to="none",
    seed=SEED,
)
# Intha transformers version support pannaatha args-a drop pannrom (v5-la sila args remove aayiduchu)
valid = set(inspect.signature(TrainingArguments.__init__).parameters)
print("dropped args:", [k for k in cfg if k not in valid])
args = TrainingArguments(**{k: v for k, v in cfg.items() if k in valid})

# Parameter groups: LoRA layers vs score head, thani thani learning rate
head_params = [p for n, p in model.named_parameters() if p.requires_grad and "score" in n]
lora_params = [p for n, p in model.named_parameters() if p.requires_grad and "score" not in n]
print("head tensors:", len(head_params), "| lora tensors:", len(lora_params))
opt = torch.optim.AdamW([
    {"params": lora_params, "lr": LR_LORA},
    {"params": head_params, "lr": LR_HEAD},
], weight_decay=0.0)

print("total steps:", total_steps, "| warmup:", warmup_steps)
print("NOTE: kaattura Training Loss = real loss x 4 (grad accumulation sum)")

trainer = Trainer(model=model, args=args, train_dataset=tr_ds,
                  data_collator=DataCollatorWithPadding(tok),
                  optimizers=(opt, None))            # scheduler-a Trainer create pannum

t = time.time()
trainer.train()
train_minutes = (time.time() - t) / 60
print(f"training time: {train_minutes:.1f} min")

In [ ]:
from sklearn.metrics import log_loss

pred  = trainer.predict(val_ds)
probs = torch.softmax(torch.tensor(pred.predictions, dtype=torch.float32), -1).numpy()
val_ll = log_loss(val_df["label"], probs, labels=[0, 1, 2])
print(f"VAL LOG LOSS: {val_ll:.4f}   (dummy 1/3 baseline = 1.0986)")

In [ ]:
import shutil

# LoRA-va base model-oda merge panni, fp16-la save (inference-ku ready)
merged = model.merge_and_unload().half()
merged.save_pretrained(f"{OUT}/qwen-cls", safe_serialization=True)
tok.save_pretrained(f"{OUT}/qwen-cls")

info = dict(model_id=MODEL_ID, n_train=N_TRAIN, n_val=N_VAL, max_p=MAX_P, max_r=MAX_R,
            lora_r=16, lr_lora=LR_LORA, lr_head=LR_HEAD, epochs=1, train_minutes=round(train_minutes, 1), val_log_loss=round(val_ll, 4))
json.dump(info, open(f"{OUT}/train_info.json", "w"), indent=2)
print(info)

shutil.rmtree(f"{OUT}/ckpt", ignore_errors=True)   # output size kuraika
!du -sh /kaggle/working/*